Diagnose WMG API response structure and find valid Census version

In [1]:

import requests, json, cellxgene_census

# 1. Check WMG API response structure
r = requests.get("https://api.cellxgene.cziscience.com/wmg/v2/primary_filter_dimensions", timeout=60)
data = r.json()
print("Top-level keys:", list(data.keys()))
print("Snapshot ID:", data.get("snapshot_id"))

# Inspect genes structure
genes_raw = data.get("genes", {})
print("\ngenes type:", type(genes_raw))
if isinstance(genes_raw, dict):
    print("genes keys:", list(genes_raw.keys())[:5])
    hs = genes_raw.get("Homo sapiens", genes_raw.get("NCBITaxon:9606", []))
    print(f"Homo sapiens entries: {len(hs)}")
    if hs:
        print("First 3 entries:", hs[:3])
elif isinstance(genes_raw, list):
    print(f"genes list length: {len(genes_raw)}")
    print("First 3:", genes_raw[:3])

# 2. Check valid Census versions
print("\n--- Valid Census versions ---")
versions = cellxgene_census.get_census_version_directory()
for v in list(versions.keys())[:10]:
    print(v, versions[v].get("release_date",""))


Top-level keys: ['gene_terms', 'organism_terms', 'snapshot_id', 'tissue_terms']
Snapshot ID: 1762972271

genes type: <class 'dict'>
genes keys: []
Homo sapiens entries: 0

--- Valid Census versions ---
stable None
latest None
2025-11-17 None
2025-11-10 None
2025-11-08 None
2025-11-04 None
2025-01-30 None
2024-07-01 None
2024-05-20 None
2023-12-15 None


Inspect new WMG API schema - gene_terms and tissue_terms structure

In [3]:

import requests, json

r = requests.get("https://api.cellxgene.cziscience.com/wmg/v2/primary_filter_dimensions", timeout=60)
data = r.json()

# Inspect gene_terms
gene_terms = data.get("gene_terms", {})
print("gene_terms type:", type(gene_terms))
print("gene_terms keys:", list(gene_terms.keys())[:5])

# Try to find human genes
for k in gene_terms.keys():
    print(f"  key='{k}', len={len(gene_terms[k])}, sample={gene_terms[k][:2]}")


gene_terms type: <class 'dict'>
gene_terms keys: ['NCBITaxon:10090', 'NCBITaxon:9606']
  key='NCBITaxon:10090', len=49413, sample=[{'ENSMUSG00000000001': 'Gnai3'}, {'ENSMUSG00000000003': 'Pbsn'}]
  key='NCBITaxon:9606', len=60810, sample=[{'ENSG00000000003': 'TSPAN6'}, {'ENSG00000000005': 'TNMD'}]


Test WMG query with new schema and check Census version + organism terms

In [5]:

import requests, json, cellxgene_census

# 1. Parse gene_terms correctly
r = requests.get("https://api.cellxgene.cziscience.com/wmg/v2/primary_filter_dimensions", timeout=60)
data = r.json()

gene_terms = data["gene_terms"]["NCBITaxon:9606"]  # list of {ENSG_id: symbol} dicts
wmg_genes = {}
for entry in gene_terms:
    wmg_genes.update(entry)  # each entry is a single-key dict
print(f"Human genes parsed: {len(wmg_genes)}")
print("Sample:", list(wmg_genes.items())[:3])

# 2. Check organism_terms
print("\norganism_terms:", data.get("organism_terms", [])[:5])

# 3. Test a small WMG query with 3 well-known genes
test_genes = ["ENSG00000000003", "ENSG00000000005", "ENSG00000000419"]  # TSPAN6, TNMD, DPM1
payload = {
    "snapshot_id": data["snapshot_id"],
    "gene_ontology_term_ids": test_genes,
    "organism_ontology_term_id": "NCBITaxon:9606",
    "filters": {
        "disease_ontology_term_ids": ["PATO:0000461"],
        "self_reported_ethnicity_ontology_term_ids": [],
        "sex_ontology_term_ids": [],
        "publication_citations": [],
    },
}
r2 = requests.post("https://api.cellxgene.cziscience.com/wmg/v2/query", json=payload, timeout=120)
print(f"\nQuery status: {r2.status_code}")
resp = r2.json()
print("Response keys:", list(resp.keys()))

# Inspect expression_summary structure
expr = resp.get("expression_summary", {})
print(f"Genes in response: {len(expr)}")
for gid, tissues in list(expr.items())[:1]:
    print(f"\nGene {gid} ({wmg_genes.get(gid,'?')}):")
    for tid, cts in list(tissues.items())[:2]:
        print(f"  tissue {tid}: {len(cts)} cell types")
        for ctid, vals in list(cts.items())[:2]:
            print(f"    {ctid}: {vals}")

# 4. Latest valid Census version
print("\n--- Census versions ---")
versions = cellxgene_census.get_census_version_directory()
valid = [v for v in versions if v not in ("stable","latest")]
print("Most recent:", valid[:3])


Human genes parsed: 60810
Sample: [('ENSG00000000003', 'TSPAN6'), ('ENSG00000000005', 'TNMD'), ('ENSG00000000419', 'DPM1')]

organism_terms: [{'NCBITaxon:10090': 'Mus musculus'}, {'NCBITaxon:9606': 'Homo sapiens'}]

Query status: 400
Response keys: ['detail', 'status', 'title', 'type']
Genes in response: 0

--- Census versions ---
Most recent: ['2025-11-17', '2025-11-10', '2025-11-08']


Debug WMG 400 error and find correct filter schema

In [7]:

import requests, json

r = requests.get("https://api.cellxgene.cziscience.com/wmg/v2/primary_filter_dimensions", timeout=60)
data = r.json()
snapshot_id = data["snapshot_id"]

# Print full error detail
test_genes = ["ENSG00000000003", "ENSG00000000005", "ENSG00000000419"]
payload = {
    "snapshot_id": snapshot_id,
    "gene_ontology_term_ids": test_genes,
    "organism_ontology_term_id": "NCBITaxon:9606",
    "filters": {
        "disease_ontology_term_ids": ["PATO:0000461"],
        "self_reported_ethnicity_ontology_term_ids": [],
        "sex_ontology_term_ids": [],
        "publication_citations": [],
    },
}
r2 = requests.post("https://api.cellxgene.cziscience.com/wmg/v2/query", json=payload, timeout=120)
print("Status:", r2.status_code)
print("Error:", json.dumps(r2.json(), indent=2))


Status: 400
Error: {
  "detail": "'filter' is a required property",
  "status": 400,
  "title": "Bad Request",
  "type": "about:blank"
}


Test WMG query with corrected filter key (singular)

In [9]:

import requests, json

r = requests.get("https://api.cellxgene.cziscience.com/wmg/v2/primary_filter_dimensions", timeout=60)
data = r.json()
snapshot_id = data["snapshot_id"]

test_genes = ["ENSG00000000003", "ENSG00000000005", "ENSG00000000419"]

# Try with singular 'filter'
payload = {
    "snapshot_id": snapshot_id,
    "gene_ontology_term_ids": test_genes,
    "organism_ontology_term_id": "NCBITaxon:9606",
    "filter": {
        "disease_ontology_term_ids": ["PATO:0000461"],
        "self_reported_ethnicity_ontology_term_ids": [],
        "sex_ontology_term_ids": [],
        "publication_citations": [],
    },
}
r2 = requests.post("https://api.cellxgene.cziscience.com/wmg/v2/query", json=payload, timeout=120)
print("Status:", r2.status_code)
resp = r2.json()
print("Response keys:", list(resp.keys()))

if r2.status_code == 200:
    expr = resp.get("expression_summary", {})
    print(f"Genes in response: {len(expr)}")
    term_labels = resp.get("term_id_labels", {})
    for gid, tissues in list(expr.items())[:1]:
        print(f"\nGene {gid}:")
        for tid, cts in list(tissues.items())[:2]:
            print(f"  tissue '{term_labels.get(tid, tid)}': {len(cts)} cell types")
            for ctid, vals in list(cts.items())[:3]:
                print(f"    {term_labels.get(ctid, ctid)}: {vals}")
else:
    print("Error:", json.dumps(resp, indent=2))


Status: 400
Response keys: ['detail', 'status', 'title', 'type']
Error: {
  "detail": "'gene_ontology_term_ids' is a required property - 'filter'",
  "status": 400,
  "title": "Bad Request",
  "type": "about:blank"
}


Test corrected WMG query schema - genes inside filter object

In [11]:

import requests, json, time

r = requests.get("https://api.cellxgene.cziscience.com/wmg/v2/primary_filter_dimensions", timeout=60)
data = r.json()
snapshot_id = data["snapshot_id"]

# Parse gene list correctly
gene_terms = data["gene_terms"]["NCBITaxon:9606"]
wmg_genes = {}
for entry in gene_terms:
    wmg_genes.update(entry)
print(f"Human genes: {len(wmg_genes)}")

# Correct schema: genes + organism go INSIDE filter
test_genes = ["ENSG00000000003", "ENSG00000000005", "ENSG00000000419"]
payload = {
    "filter": {
        "gene_ontology_term_ids": test_genes,
        "organism_ontology_term_id": "NCBITaxon:9606",
        "disease_ontology_term_ids": ["PATO:0000461"],
        "self_reported_ethnicity_ontology_term_ids": [],
        "sex_ontology_term_ids": [],
        "publication_citations": [],
    },
    "is_rollup": True,
}

t0 = time.time()
r2 = requests.post("https://api.cellxgene.cziscience.com/wmg/v2/query", json=payload, timeout=120)
print(f"Status: {r2.status_code} ({time.time()-t0:.1f}s)")

resp = r2.json()
print("Response keys:", list(resp.keys()))

expr = resp.get("expression_summary", {})
print(f"Genes in response: {len(expr)}")

term_labels = resp.get("term_id_labels", {})
cell_types_by_tissue = term_labels.get("cell_types", {})

for gid, tissues in list(expr.items())[:1]:
    print(f"\nGene {gid} ({wmg_genes.get(gid,'?')}):")
    for tid, cts in list(tissues.items())[:2]:
        # Get tissue name from cell_types label structure
        tissue_info = cell_types_by_tissue.get(tid, {})
        tissue_stats = tissue_info.get("tissue_stats", {}).get("aggregated", {})
        tissue_name = tissue_stats.get("name", tid)
        n_cts = len([k for k in cts if k != "tissue_stats"])
        print(f"  tissue '{tissue_name}': {n_cts} cell types")
        for ctid, ct_vals in list(cts.items())[:3]:
            if ctid == "tissue_stats":
                continue
            agg = ct_vals.get("aggregated", {})
            ct_name = cell_types_by_tissue.get(tid, {}).get(ctid, {}).get("aggregated", {}).get("name", ctid)
            print(f"    {ct_name}: me={agg.get('me',0):.3f}, n={agg.get('n',0)}, pc={agg.get('pc',0):.3f}")

# Count total records
total = sum(
    len([k for k in cts if k != "tissue_stats"])
    for tissues in expr.values()
    for cts in tissues.values()
)
print(f"\nTotal gene×tissue×celltype records: {total}")


Human genes: 60810
Status: 200 (1.1s)
Response keys: ['expression_summary', 'snapshot_id', 'term_id_labels']
Genes in response: 3

Gene ENSG00000000003 (TSPAN6):
  tissue 'nose': 104 cell types
    cell: me=1.904, n=37383, pc=0.171
    CL:0000034: me=1.778, n=2528, pc=0.103
    CL:0000036: me=1.778, n=2528, pc=0.103
  tissue 'peripheral nervous system': 8 cell types
    cell: me=1.460, n=3, pc=0.007
    epithelial cell: me=1.439, n=1, pc=0.004
    CL:0000075: me=1.439, n=1, pc=0.004

Total gene×tissue×celltype records: 10959


Check rollup behavior and cell type label parsing, then benchmark 200-gene batch

In [13]:

import requests, json, time

r = requests.get("https://api.cellxgene.cziscience.com/wmg/v2/primary_filter_dimensions", timeout=60)
data = r.json()
snapshot_id = data["snapshot_id"]

gene_terms = data["gene_terms"]["NCBITaxon:9606"]
wmg_genes = {}
for entry in gene_terms:
    wmg_genes.update(entry)

# Test with is_rollup=False to get leaf cell types only
test_genes = ["ENSG00000000003"]
for rollup in [True, False]:
    payload = {
        "filter": {
            "gene_ontology_term_ids": test_genes,
            "organism_ontology_term_id": "NCBITaxon:9606",
            "disease_ontology_term_ids": ["PATO:0000461"],
            "self_reported_ethnicity_ontology_term_ids": [],
            "sex_ontology_term_ids": [],
            "publication_citations": [],
        },
        "is_rollup": rollup,
    }
    r2 = requests.post("https://api.cellxgene.cziscience.com/wmg/v2/query", json=payload, timeout=120)
    resp = r2.json()
    expr = resp.get("expression_summary", {})
    term_labels = resp.get("term_id_labels", {})
    cell_types_by_tissue = term_labels.get("cell_types", {})

    # Count unique cell type IDs across all tissues
    all_ct_ids = set()
    for gid, tissues in expr.items():
        for tid, cts in tissues.items():
            for ctid in cts:
                if ctid != "tissue_stats":
                    all_ct_ids.add(ctid)

    # Check if "cell" (CL:0000000) appears
    has_generic_cell = "cell" in all_ct_ids or "CL:0000000" in all_ct_ids
    print(f"is_rollup={rollup}: {len(all_ct_ids)} unique cell types, has 'cell' root: {has_generic_cell}")
    print(f"  Sample ct IDs: {list(all_ct_ids)[:8]}")

# Now check cell type label resolution
print("\n--- Label resolution for nose tissue ---")
payload = {
    "filter": {
        "gene_ontology_term_ids": ["ENSG00000000003"],
        "organism_ontology_term_id": "NCBITaxon:9606",
        "disease_ontology_term_ids": ["PATO:0000461"],
        "self_reported_ethnicity_ontology_term_ids": [],
        "sex_ontology_term_ids": [],
        "publication_citations": [],
    },
    "is_rollup": True,
}
r2 = requests.post("https://api.cellxgene.cziscience.com/wmg/v2/query", json=payload, timeout=120)
resp = r2.json()
term_labels = resp["term_id_labels"]
cell_types_by_tissue = term_labels["cell_types"]

# Find nose tissue
for tid, cts in cell_types_by_tissue.items():
    ts = cts.get("tissue_stats", {}).get("aggregated", {})
    if ts.get("name") == "nose":
        print(f"Tissue: {tid} = nose, total_count={ts.get('total_count')}")
        for ctid, ct_data in list(cts.items())[:6]:
            if ctid == "tissue_stats": continue
            agg = ct_data.get("aggregated", {})
            print(f"  {ctid} -> name='{agg.get('name')}', total_count={agg.get('total_count')}")
        break

# Benchmark 200-gene batch
print("\n--- Benchmarking 200-gene batch ---")
batch_200 = list(wmg_genes.keys())[:200]
payload["filter"]["gene_ontology_term_ids"] = batch_200
t0 = time.time()
r3 = requests.post("https://api.cellxgene.cziscience.com/wmg/v2/query", json=payload, timeout=180)
elapsed = time.time() - t0
resp3 = r3.json()
expr3 = resp3.get("expression_summary", {})
total_records = sum(
    len([k for k in cts if k != "tissue_stats"])
    for tissues in expr3.values()
    for cts in tissues.values()
)
print(f"200 genes: {elapsed:.1f}s, {total_records:,} records, {200/elapsed:.1f} genes/s")


is_rollup=True: 780 unique cell types, has 'cell' root: True
  Sample ct IDs: ['CL:4023041', 'CL:4023068', 'CL:0000081', 'CL:0000723', 'CL:1000143', 'CL:0008036', 'CL:4030026', 'CL:0000818']
is_rollup=False: 0 unique cell types, has 'cell' root: False
  Sample ct IDs: []

--- Label resolution for nose tissue ---
Tissue: UBERON:0000004 = nose, total_count=219562
  CL:0000000 -> name='cell', total_count=219562
  CL:0000064 -> name='ciliated cell', total_count=27121
  CL:0000066 -> name='epithelial cell', total_count=165124
  CL:0000068 -> name='duct epithelial cell', total_count=491
  CL:0000071 -> name='blood vessel endothelial cell', total_count=144
  CL:0000076 -> name='squamous epithelial cell', total_count=1173

--- Benchmarking 200-gene batch ---
200 genes: 19.9s, 767,179 records, 10.0 genes/s


Aggregate WMG raw records into cell_type × gene matrices

In [15]:

import pandas as pd
import numpy as np
import logging, sys, time
from pathlib import Path

OUT_DIR = Path("/mnt/results/01_cellxgene_pseudobulk")
TMP_DIR = OUT_DIR / "tmp"

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(message)s",
                    datefmt="%H:%M:%S", stream=sys.stdout)
log = logging.getLogger(__name__)

# ── Load raw data ──────────────────────────────────────────────────────────────
t0 = time.time()
log.info("Loading raw parquet...")
df = pd.read_parquet(TMP_DIR / "wmg_raw_all.parquet")
log.info(f"  {len(df):,} records loaded in {time.time()-t0:.1f}s")
log.info(f"  Genes: {df['gene_id'].nunique():,} | Tissues: {df['tissue_id'].nunique()} | Cell types: {df['cell_type_id'].nunique()}")
print(df.dtypes)
print(df.head(3))


[00:54:08] Loading raw parquet...
[00:54:26]   90,694,622 records loaded in 17.7s
[00:54:33]   Genes: 60,621 | Tissues: 55 | Cell types: 871
gene_id               object
gene_symbol           object
tissue_id             object
tissue_label          object
tissue_total_n         int64
cell_type_id          object
cell_type_label       object
cell_type_total_n      int64
mean_expr            float64
n_expr                 int64
frac_expr            float64
frac_of_tissue       float64
dtype: object
           gene_id gene_symbol  ... frac_expr frac_of_tissue
0  ENSG00000000003      TSPAN6  ...  0.103074       0.011514
1  ENSG00000000003      TSPAN6  ...  0.103074       0.011514
2  ENSG00000000003      TSPAN6  ...  0.120000       0.000027

[3 rows x 12 columns]


Aggregate across tissues weighted by cell_type_total_n, build pivot matrices

In [17]:

import pandas as pd
import numpy as np
import time
from pathlib import Path

OUT_DIR = Path("/mnt/results/01_cellxgene_pseudobulk")
TMP_DIR = OUT_DIR / "tmp"

t0 = time.time()

# ── Inspect duplicates first ───────────────────────────────────────────────────
# Check if same (gene, tissue, cell_type) appears multiple times
dup_check = df.groupby(["gene_id", "tissue_id", "cell_type_id"]).size()
print(f"Max records per (gene, tissue, cell_type): {dup_check.max()}")
print(f"Any duplicates: {(dup_check > 1).sum()} triplets have >1 record")

# ── Aggregate across tissues: weighted mean by cell_type_total_n ───────────────
# cell_type_total_n = total cells of that cell type in that tissue (denominator)
# This is the correct weight: larger tissue contributions dominate
print("\nAggregating across tissues...")

# Use cell_type_total_n as weight (clip to 1 to avoid zero-weight)
df["weight"] = df["cell_type_total_n"].clip(lower=1)

agg = (
    df.groupby(["gene_id", "gene_symbol", "cell_type_id", "cell_type_label"])
    .apply(
        lambda g: pd.Series({
            "mean_expr":   np.average(g["mean_expr"],  weights=g["weight"]),
            "frac_expr":   np.average(g["frac_expr"],  weights=g["weight"]),
            "total_cells": g["cell_type_total_n"].max(),   # max across tissues = total unique cells
            "n_tissues":   g["tissue_id"].nunique(),
        }),
        include_groups=False
    )
    .reset_index()
)

print(f"Aggregated in {time.time()-t0:.1f}s")
print(f"(gene × cell_type) pairs: {len(agg):,}")
print(f"Unique genes: {agg['gene_id'].nunique():,}")
print(f"Unique cell types: {agg['cell_type_id'].nunique():,}")
print(agg.head(3))


Execution timed out after 60 minutes. Your code took too long to complete.

Tips:
- Break down long-running operations into smaller steps
- For HPC tools (AlphaFold, Boltz, etc.), the timeout is automatically extended to 2 hours
- Workspace state has been reset due to timeout

Fast vectorized aggregation using numpy - weighted sum then divide

In [19]:

import pandas as pd
import numpy as np
import time
from pathlib import Path

OUT_DIR = Path("/mnt/results/01_cellxgene_pseudobulk")
TMP_DIR = OUT_DIR / "tmp"

t0 = time.time()
print("Loading raw parquet...")
df = pd.read_parquet(TMP_DIR / "wmg_raw_all.parquet")
print(f"  {len(df):,} records in {time.time()-t0:.1f}s")

# ── Vectorized weighted aggregation ───────────────────────────────────────────
# weight = cell_type_total_n (total cells of that type in that tissue)
print("\nComputing weighted aggregation...")
t1 = time.time()

df["weight"] = df["cell_type_total_n"].clip(lower=1).astype(np.float32)
df["me_w"]   = (df["mean_expr"] * df["weight"]).astype(np.float32)
df["fe_w"]   = (df["frac_expr"] * df["weight"]).astype(np.float32)

grp_cols = ["gene_id", "gene_symbol", "cell_type_id", "cell_type_label"]

agg = df.groupby(grp_cols, sort=False).agg(
    me_w_sum   = ("me_w",   "sum"),
    fe_w_sum   = ("fe_w",   "sum"),
    w_sum      = ("weight", "sum"),
    n_tissues  = ("tissue_id", "nunique"),
    total_cells= ("cell_type_total_n", "max"),
).reset_index()

agg["mean_expr"] = agg["me_w_sum"] / agg["w_sum"]
agg["frac_expr"] = agg["fe_w_sum"] / agg["w_sum"]
agg.drop(columns=["me_w_sum","fe_w_sum","w_sum"], inplace=True)

print(f"  Done in {time.time()-t1:.1f}s")
print(f"  (gene × cell_type) pairs: {len(agg):,}")
print(f"  Unique genes:      {agg['gene_id'].nunique():,}")
print(f"  Unique cell types: {agg['cell_type_id'].nunique():,}")
print(agg.head(3))


Loading raw parquet...
  90,694,622 records in 17.7s

Computing weighted aggregation...
  Done in 67.0s
  (gene × cell_type) pairs: 24,457,371
  Unique genes:      60,621
  Unique cell types: 871
           gene_id gene_symbol cell_type_id  ... total_cells  mean_expr  frac_expr
0  ENSG00000000003      TSPAN6   CL:0000034  ...           0   1.723243   0.095793
1  ENSG00000000003      TSPAN6   CL:0000036  ...           0   1.810503   0.136475
2  ENSG00000000003      TSPAN6   CL:0000057  ...           0   2.114001   0.120000

[3 rows x 8 columns]


Build pivot matrices, quality filter, and save final outputs

In [21]:

import pandas as pd
import numpy as np
import time
from pathlib import Path

OUT_DIR = Path("/mnt/results/01_cellxgene_pseudobulk")
TMP_DIR = OUT_DIR / "tmp"

# ── Cell type metadata ─────────────────────────────────────────────────────────
print("Building cell type metadata...")
ct_meta = (
    agg[["cell_type_id", "cell_type_label", "total_cells", "n_tissues"]]
    .groupby(["cell_type_id", "cell_type_label"], sort=False)
    .agg(total_cells=("total_cells", "max"), n_tissues=("n_tissues", "max"))
    .reset_index()
    .sort_values("total_cells", ascending=False)
)
print(f"  Total cell types: {len(ct_meta)}")
print(ct_meta.head(10).to_string(index=False))

# ── Quality filter: keep cell types with total_cells >= 50 ────────────────────
MIN_CELLS = 50
keep_ct = set(ct_meta.loc[ct_meta["total_cells"] >= MIN_CELLS, "cell_type_id"])
print(f"\nCell types with >= {MIN_CELLS} cells: {len(keep_ct)}")

agg_filt = agg[agg["cell_type_id"].isin(keep_ct)].copy()
print(f"Filtered pairs: {len(agg_filt):,}")

# ── Build pivot matrices ───────────────────────────────────────────────────────
print("\nBuilding mean expression pivot...")
t0 = time.time()

# Use gene_symbol as columns, cell_type_label as index
mean_mat = agg_filt.pivot_table(
    index="cell_type_label", columns="gene_symbol",
    values="mean_expr", fill_value=0.0, aggfunc="mean"
)
print(f"  Mean matrix: {mean_mat.shape} in {time.time()-t0:.1f}s")

print("Building fraction expressing pivot...")
t0 = time.time()
frac_mat = agg_filt.pivot_table(
    index="cell_type_label", columns="gene_symbol",
    values="frac_expr", fill_value=0.0, aggfunc="mean"
)
print(f"  Frac matrix: {frac_mat.shape} in {time.time()-t0:.1f}s")

# Log1p
log1p_mat = np.log1p(mean_mat)

# ── Summary stats ──────────────────────────────────────────────────────────────
print(f"\nFinal matrix dimensions: {mean_mat.shape[0]} cell types × {mean_mat.shape[1]} genes")
print(f"Mean expression range:   {mean_mat.values.min():.4f} – {mean_mat.values.max():.4f}")
print(f"Fraction expr range:     {frac_mat.values.min():.4f} – {frac_mat.values.max():.4f}")
sparsity = (mean_mat.values == 0).mean() * 100
print(f"Sparsity:                {sparsity:.1f}% zeros")


Building cell type metadata...
  Total cell types: 1067
cell_type_id          cell_type_label  total_cells  n_tissues
  CL:0002319              neural cell     14911036         29
  CL:0000988       hematopoietic cell     13413551         41
  CL:0000738                leukocyte     12882395         50
  CL:0000842         mononuclear cell     12848111         48
  CL:0000542               lymphocyte     10274976         48
  CL:0000540                   neuron      9583381         27
  CL:0000084                   T cell      7604473         46
  CL:0002419            mature T cell      7401929         28
  CL:0000789        alpha-beta T cell      7172236         16
  CL:0000791 mature alpha-beta T cell      7162592         37

Cell types with >= 50 cells: 821
Filtered pairs: 24,040,877

Building mean expression pivot...
  Mean matrix: (1016, 60621) in 18.7s
Building fraction expressing pivot...
  Frac matrix: (1016, 60621) in 21.0s

Final matrix dimensions: 1016 cell types × 60621 ge

Save all final output matrices and metadata to results

In [23]:

import time
from pathlib import Path

OUT_DIR = Path("/mnt/results/01_cellxgene_pseudobulk")

# ── Save matrices ──────────────────────────────────────────────────────────────
print("Saving mean expression matrix...")
t0 = time.time()
mean_mat.to_csv(OUT_DIR / "celltype_mean_expression.csv.gz", compression="gzip")
print(f"  Done in {time.time()-t0:.1f}s")

print("Saving fraction expressing matrix...")
t0 = time.time()
frac_mat.to_csv(OUT_DIR / "celltype_fraction_expressing.csv.gz", compression="gzip")
print(f"  Done in {time.time()-t0:.1f}s")

print("Saving log1p mean expression matrix...")
t0 = time.time()
log1p_mat.to_csv(OUT_DIR / "celltype_log1p_mean_expression.csv.gz", compression="gzip")
print(f"  Done in {time.time()-t0:.1f}s")

# ── Save metadata ──────────────────────────────────────────────────────────────
print("Saving cell type metadata...")
# Add index info: which cell types are in the matrix
ct_meta_out = ct_meta[ct_meta["cell_type_id"].isin(keep_ct)].copy()
ct_meta_out["in_matrix"] = ct_meta_out["cell_type_label"].isin(mean_mat.index)
ct_meta_out.to_csv(OUT_DIR / "celltype_metadata.csv", index=False)
print(f"  {len(ct_meta_out)} cell types saved")

# Gene metadata from aggregated data
print("Saving gene metadata...")
gene_meta = (
    agg_filt[["gene_id", "gene_symbol"]]
    .drop_duplicates()
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)
gene_meta.to_csv(OUT_DIR / "gene_metadata.csv", index=False)
print(f"  {len(gene_meta):,} genes saved")

# ── File sizes ─────────────────────────────────────────────────────────────────
print("\n=== Output files ===")
for f in sorted(OUT_DIR.glob("*.csv*")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}: {size_mb:.1f} MB")


Saving mean expression matrix...
  Done in 214.3s
Saving fraction expressing matrix...
  Done in 223.4s
Saving log1p mean expression matrix...
  Done in 224.0s
Saving cell type metadata...
  1016 cell types saved
Saving gene metadata...
  60,621 genes saved

=== Output files ===
  celltype_fraction_expressing.csv.gz: 90.0 MB
  celltype_log1p_mean_expression.csv.gz: 104.6 MB
  celltype_mean_expression.csv.gz: 107.4 MB
  celltype_metadata.csv: 0.0 MB
  gene_metadata.csv: 1.6 MB


Validate matrices and check known marker gene expression

In [25]:

import pandas as pd
import numpy as np
from pathlib import Path

OUT_DIR = Path("/mnt/results/01_cellxgene_pseudobulk")

# ── Reload matrices ────────────────────────────────────────────────────────────
print("Loading matrices for validation...")
mean_mat  = pd.read_csv(OUT_DIR / "celltype_mean_expression.csv.gz", index_col=0)
frac_mat  = pd.read_csv(OUT_DIR / "celltype_fraction_expressing.csv.gz", index_col=0)
ct_meta   = pd.read_csv(OUT_DIR / "celltype_metadata.csv")

print(f"Mean matrix:  {mean_mat.shape}")
print(f"Frac matrix:  {frac_mat.shape}")
print(f"Cell types:   {len(ct_meta)}")

# ── Check known marker genes ───────────────────────────────────────────────────
# Well-established cell type markers
markers = {
    "T cell":          ["CD3D", "CD3E", "CD3G", "CD4", "CD8A"],
    "B cell":          ["CD19", "MS4A1", "CD79A", "CD79B"],
    "NK cell":         ["NCAM1", "KLRD1", "NKG7", "GNLY"],
    "Monocyte":        ["CD14", "LYZ", "CSF1R", "FCGR3A"],
    "Dendritic cell":  ["ITGAX", "CLEC9A", "FCER1A"],
    "Neutrophil":      ["S100A8", "S100A9", "FCGR3B"],
    "Erythrocyte":     ["HBA1", "HBA2", "HBB"],
    "Platelet":        ["PPBP", "PF4", "GP1BA"],
    "Neuron":          ["RBFOX3", "SYP", "MAP2", "SNAP25"],
    "Astrocyte":       ["GFAP", "AQP4", "S100B"],
    "Oligodendrocyte": ["MBP", "MOG", "PLP1"],
    "Hepatocyte":      ["ALB", "APOB", "CYP3A4"],
    "Cardiomyocyte":   ["TNNT2", "MYH7", "ACTC1"],
    "Epithelial":      ["EPCAM", "KRT8", "KRT18"],
    "Endothelial":     ["PECAM1", "VWF", "CDH5"],
    "Fibroblast":      ["COL1A1", "COL1A2", "DCN"],
}

all_markers = [g for genes in markers.values() for g in genes]
present = [g for g in all_markers if g in mean_mat.columns]
missing = [g for g in all_markers if g not in mean_mat.columns]
print(f"\nMarker genes present: {len(present)}/{len(all_markers)}")
if missing:
    print(f"Missing: {missing}")

# ── Check expression of key markers in expected cell types ────────────────────
print("\n=== Marker gene validation ===")
checks = [
    ("CD3D",  "T cell"),
    ("CD19",  "B cell"),
    ("HBB",   "erythrocyte"),
    ("MBP",   "oligodendrocyte"),
    ("ALB",   "hepatocyte"),
    ("PECAM1","endothelial cell"),
    ("COL1A1","fibroblast"),
    ("GFAP",  "astrocyte"),
]

for gene, expected_ct in checks:
    if gene not in mean_mat.columns:
        print(f"  {gene}: NOT IN MATRIX")
        continue
    col = mean_mat[gene].sort_values(ascending=False)
    top3 = col.head(3)
    # Find expected cell type (partial match)
    matches = [ct for ct in mean_mat.index if expected_ct.lower() in ct.lower()]
    if matches:
        val_expected = mean_mat.loc[matches[0], gene]
        rank = (col > val_expected).sum() + 1
        print(f"  {gene:8s}: top cell type = '{top3.index[0]}' ({top3.iloc[0]:.3f}) | "
              f"'{matches[0]}' rank={rank}/{len(col)} ({val_expected:.3f})")
    else:
        print(f"  {gene:8s}: top = '{top3.index[0]}' ({top3.iloc[0]:.3f}) | '{expected_ct}' not found")


Loading matrices for validation...
Mean matrix:  (1016, 60621)
Frac matrix:  (1016, 60621)
Cell types:   1016

Marker genes present: 54/54

=== Marker gene validation ===
  CD3D    : top cell type = 'CD8-alpha-beta-positive, alpha-beta intraepithelial T cell' (3.115) | 'CD4-negative, CD8-negative, alpha-beta intraepithelial T cell' rank=34/1016 (2.400)
  CD19    : top cell type = 'taste receptor cell' (2.515) | 'B cell' rank=246/1016 (1.795)
  HBB     : top cell type = 'CL:0000558' (8.676) | 'enucleate erythrocyte' rank=24/1016 (4.172)
  MBP     : top cell type = 'oligodendrocyte' (3.481) | 'differentiation-committed oligodendrocyte precursor' rank=6/1016 (3.188)
  ALB     : top cell type = 'hepatoblast' (5.496) | 'centrilobular region hepatocyte' rank=6/1016 (4.757)
  PECAM1  : top cell type = 'tongue muscle cell' (3.697) | 'alveolar capillary type 1 endothelial cell' rank=10/1016 (2.960)
  COL1A1  : top cell type = 'keratocyte' (4.904) | 'alveolar type 1 fibroblast cell' rank=239/101

Investigate rollup artifacts - check suspicious high-expression cell types

In [27]:

import pandas as pd
import numpy as np
from pathlib import Path

OUT_DIR = Path("/mnt/results/01_cellxgene_pseudobulk")
mean_mat = pd.read_csv(OUT_DIR / "celltype_mean_expression.csv.gz", index_col=0)
ct_meta  = pd.read_csv(OUT_DIR / "celltype_metadata.csv")

# Check the suspicious cell types
suspicious = ["taste receptor cell", "tongue muscle cell", "small intestine goblet cell", "keratocyte"]

print("=== Suspicious cell types - CD19, PECAM1, GFAP, COL1A1 expression ===")
for ct in suspicious:
    if ct in mean_mat.index:
        cd19  = mean_mat.loc[ct, "CD19"]   if "CD19"   in mean_mat.columns else "N/A"
        pecam = mean_mat.loc[ct, "PECAM1"] if "PECAM1" in mean_mat.columns else "N/A"
        gfap  = mean_mat.loc[ct, "GFAP"]   if "GFAP"   in mean_mat.columns else "N/A"
        col1  = mean_mat.loc[ct, "COL1A1"] if "COL1A1" in mean_mat.columns else "N/A"
        meta_row = ct_meta[ct_meta["cell_type_label"] == ct]
        n = meta_row["total_cells"].values[0] if len(meta_row) else "?"
        nt = meta_row["n_tissues"].values[0] if len(meta_row) else "?"
        print(f"\n  {ct} (n_cells={n}, n_tissues={nt}):")
        print(f"    CD19={cd19:.3f}, PECAM1={pecam:.3f}, GFAP={gfap:.3f}, COL1A1={col1:.3f}")

# Check n_tissues distribution - rollup parents appear in many tissues
print("\n=== n_tissues distribution ===")
print(ct_meta["n_tissues"].describe())
print("\nCell types appearing in ALL 55 tissues (likely broad rollups):")
broad = ct_meta[ct_meta["n_tissues"] >= 50].sort_values("total_cells", ascending=False)
print(broad[["cell_type_label","cell_type_id","total_cells","n_tissues"]].head(20).to_string(index=False))


=== Suspicious cell types - CD19, PECAM1, GFAP, COL1A1 expression ===

  taste receptor cell (n_cells=4712, n_tissues=2):
    CD19=2.515, PECAM1=1.532, GFAP=2.411, COL1A1=2.265

  tongue muscle cell (n_cells=202, n_tissues=1):
    CD19=0.000, PECAM1=3.697, GFAP=2.621, COL1A1=3.458

  small intestine goblet cell (n_cells=6235, n_tissues=1):
    CD19=0.000, PECAM1=1.562, GFAP=3.026, COL1A1=1.647

  keratocyte (n_cells=219280, n_tissues=2):
    CD19=1.748, PECAM1=1.866, GFAP=1.674, COL1A1=4.904

=== n_tissues distribution ===
count    1016.000000
mean        5.826772
std         9.149094
min         1.000000
25%         1.000000
50%         2.000000
75%         6.000000
max        52.000000
Name: n_tissues, dtype: float64

Cell types appearing in ALL 55 tissues (likely broad rollups):
  cell_type_label cell_type_id  total_cells  n_tissues
        leukocyte   CL:0000738     12882395         50
myeloid leukocyte   CL:0000766      2516607         50
       macrophage   CL:0000235       49428

Generate marker gene heatmap for well-represented cell types

In [29]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUT_DIR = Path("/mnt/results/01_cellxgene_pseudobulk")
mean_mat = pd.read_csv(OUT_DIR / "celltype_mean_expression.csv.gz", index_col=0)
ct_meta  = pd.read_csv(OUT_DIR / "celltype_metadata.csv")

# ── Select representative cell types (well-known, large n) ────────────────────
selected_cts = [
    # Immune
    "T cell", "CD4-positive, alpha-beta T cell", "CD8-positive, alpha-beta T cell",
    "B cell", "plasma cell",
    "natural killer cell",
    "classical monocyte", "non-classical monocyte",
    "plasmacytoid dendritic cell", "conventional dendritic cell",
    "neutrophil", "mast cell",
    # Erythroid/Platelet
    "erythrocyte", "platelet",
    # Neural
    "neuron", "astrocyte", "oligodendrocyte", "microglial cell",
    # Structural
    "hepatocyte", "cardiomyocyte cell",
    "endothelial cell", "fibroblast",
    "epithelial cell",
]

# Match to actual index (partial match, pick best)
def find_ct(name, index):
    exact = [ct for ct in index if ct.lower() == name.lower()]
    if exact: return exact[0]
    partial = [ct for ct in index if name.lower() in ct.lower()]
    if partial: return partial[0]
    return None

ct_map = {name: find_ct(name, mean_mat.index) for name in selected_cts}
ct_map = {k: v for k, v in ct_map.items() if v is not None}
print("Matched cell types:")
for k, v in ct_map.items():
    print(f"  {k:40s} -> {v}")


Matched cell types:
  T cell                                   -> T cell
  CD4-positive, alpha-beta T cell          -> CD4-positive, alpha-beta T cell
  CD8-positive, alpha-beta T cell          -> CD8-positive, alpha-beta T cell
  B cell                                   -> B cell
  plasma cell                              -> plasma cell
  natural killer cell                      -> natural killer cell
  classical monocyte                       -> classical monocyte
  non-classical monocyte                   -> non-classical monocyte
  plasmacytoid dendritic cell              -> plasmacytoid dendritic cell
  conventional dendritic cell              -> conventional dendritic cell
  neutrophil                               -> neutrophil
  mast cell                                -> mast cell
  erythrocyte                              -> erythrocyte
  platelet                                 -> platelet
  neuron                                   -> neuron
  astrocyte                      

Generate marker gene × cell type validation heatmap

In [31]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUT_DIR = Path("/mnt/results/01_cellxgene_pseudobulk")

# Canonical markers per cell type (ordered to match selected_cts)
markers_ordered = {
    "T cell":                          ["CD3D", "CD3E", "CD3G"],
    "CD4-positive, alpha-beta T cell": ["CD4", "IL7R", "FOXP3"],
    "CD8-positive, alpha-beta T cell": ["CD8A", "CD8B", "GZMB"],
    "B cell":                          ["CD19", "MS4A1", "CD79A"],
    "plasma cell":                     ["IGHG1", "MZB1", "SDC1"],
    "natural killer cell":             ["NCAM1", "NKG7", "GNLY"],
    "classical monocyte":              ["CD14", "LYZ", "S100A8"],
    "non-classical monocyte":          ["FCGR3A", "CX3CR1", "LST1"],
    "plasmacytoid dendritic cell":     ["LILRA4", "CLEC4C", "IL3RA"],
    "conventional dendritic cell":     ["ITGAX", "FCER1A", "CLEC9A"],
    "neutrophil":                      ["S100A9", "FCGR3B", "CXCR2"],
    "mast cell":                       ["TPSAB1", "KIT", "CPA3"],
    "erythrocyte":                     ["HBB", "HBA1", "GYPA"],
    "platelet":                        ["PPBP", "PF4", "GP1BA"],
    "neuron":                          ["RBFOX3", "SYP", "SNAP25"],
    "astrocyte":                       ["GFAP", "AQP4", "S100B"],
    "oligodendrocyte":                 ["MBP", "MOG", "PLP1"],
    "microglial cell":                 ["CX3CR1", "P2RY12", "TMEM119"],
    "hepatocyte":                      ["ALB", "APOB", "CYP3A4"],
    "endothelial cell":                ["PECAM1", "VWF", "CDH5"],
    "fibroblast":                      ["COL1A1", "COL1A2", "DCN"],
    "epithelial cell":                 ["EPCAM", "KRT8", "KRT18"],
}

# Build submatrix: rows = cell types, cols = marker genes
ct_labels  = list(ct_map.values())   # actual index labels
gene_list  = []
for genes in markers_ordered.values():
    for g in genes:
        if g in mean_mat.columns and g not in gene_list:
            gene_list.append(g)

sub = np.log1p(mean_mat.loc[ct_labels, gene_list])

# Row-normalize (z-score per gene) for visual clarity
sub_z = sub.copy()
for col in sub_z.columns:
    mu, sd = sub_z[col].mean(), sub_z[col].std()
    sub_z[col] = (sub_z[col] - mu) / (sd + 1e-6)
sub_z = sub_z.clip(-3, 3)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 10))

sns.heatmap(
    sub_z,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-3, vmax=3,
    xticklabels=gene_list,
    yticklabels=ct_labels,
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"label": "Z-score (log1p mean expr)", "shrink": 0.6},
)

ax.set_xlabel("Marker genes", fontsize=11)
ax.set_ylabel("")
ax.set_title("Cell type marker gene expression\n(CELLxGENE WMG, 1,016 cell types × 60,621 genes, healthy human)", fontsize=12)
ax.tick_params(axis="x", labelsize=8, rotation=90)
ax.tick_params(axis="y", labelsize=8)

# Add vertical separators between cell type groups (every 3 genes)
for i in range(3, len(gene_list), 3):
    ax.axvline(i, color="black", linewidth=1.0, alpha=0.5)

plt.tight_layout()

out_png = OUT_DIR / "marker_gene_heatmap.png"
out_svg = OUT_DIR / "marker_gene_heatmap.svg"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
fig.savefig(out_svg, bbox_inches="tight")
plt.close()
print("Saved heatmap.")


Saved heatmap.
